# 09. Initialization Diagnostics（spec §14.3）

## 学習目標
- 学習前の1回のforward/backwardだけで、初期化スキームがlogit分布・活性RMS・勾配ノルムに与える影響を測る
- 「良い初期化」は経験則でなく測定で選ぶ

## 比較スキーム
- `normal_0.02`（GPT-2既定 + 残差枝の 1/√(2L) スケーリング）
- `normal_0.02_noscale`（残差スケーリングなし）
- `xavier`（Glorot）/ `kaiming`（He, relu gain）

理想の初期化は logit がほぼ一様（loss≈ln V）で、活性・勾配が層間で安定していること。

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"
CAL = ROOT / "experiments/calibrations"

In [2]:
res = load_json(CAL / "init.json")["results"]
import math
rows = []
for s, v in res.items():
    rows.append((s, v["init_loss"], v["logit_std"], v["softmax_entropy"], v["grad_norm_total"]))
df = pd.DataFrame(rows, columns=["scheme","init_loss","logit_std","softmax_entropy","grad_norm"])
print(f"理想 init_loss = ln V = {math.log(8192):.3f}")
df.round(3)

理想 init_loss = ln V = 9.011


,scheme,init_loss,logit_std,softmax_entropy,grad_norm
0,normal_0.02,9.067,0.320,8.960,3.767
1,normal_0.02_noscale,9.041,0.318,8.960,2.053
2,xavier,9.030,0.246,8.981,1.282
3,kaiming,9.977,1.411,8.023,3.033


In [3]:
# 残差ストリーム RMS の層プロファイル（爆発/消失の兆候）
fig = go.Figure()
for s, v in res.items():
    fig.add_trace(go.Scatter(y=v["resid_rms_by_layer"], mode="lines+markers", name=s))
fig.update_layout(title="初期化スキーム別 residual RMS（層プロファイル）",
                  xaxis_title="layer", yaxis_title="RMS", template="plotly_white", height=380)
fig.show()

## Observation / Interpretation / Caveat
- **Observation**: `normal_0.02` は init_loss ≈ ln V（一様に近い＝過信していない）。`kaiming` は logit_std が大きく init_loss が ln V を超える（過信）。`xavier` は最も冷たい。残差スケーリングの有無で深い層のRMSプロファイルが変わる。
- **Interpretation**: 分類ヘッド前の logit が過大だと初期損失が悪化し、序盤に大きな勾配→不安定を招きうる。GPT-2既定が「logitほぼ一様＋残差RMS安定」を両立するため既定として妥当。
- **Caveat**: これは学習前1ステップの統計。最終性能への影響は短期runで別途確認すべき（初期化の差は数百ステップで縮小することも多い）。1シード。

**次へ**: [10_hardware_calibration](10_hardware_calibration.ipynb)